# BioSNAP Dataset Preprocessing for QuVINE

This notebook helps you:

1. Download BioSNAP biological network datasets
2. Convert them into clean undirected NetworkX graphs
3. Extract repeated **connected induced subgraphs** of approximately **2k, 5k, 10k nodes**
4. Use a degree-aware sampling heuristic to better preserve the original degree distribution
5. Save sampled subgraphs as edge lists and metadata for downstream QuVINE processing

## Datasets

BioSNAP datasets include:
- **CC-Neuron**: Neuron connectivity
- **DD-Miner**: Disease-Disease associations
- **ChG-Miner**: Chemical-Gene interactions
- **ChG-InterDecagon**: Chemical-Gene interactions (Decagon)
- **DCh-Miner**: Disease-Chemical associations
- **ChSe-Decagon**: Chemical-Side effect associations
- **DG-AssocMiner**: Disease-Gene associations
- **FF-Miner**: Food-Food interactions
- **DF-Miner**: Drug-Food interactions

## Design choices

- **Loader**: Direct download from BioSNAP URLs with web scraping
- **Sampling goal**: connected, induced, degree-aware, repeated multiple times (30 replicas per size)
- **Output format**: CSV edgelists + JSON metadata

## Notes

- The degree-preservation step here is heuristic, not an exact constrained optimization.
- Subgraph sampling preserves connectivity.
- BioSNAP datasets may be bipartite; we convert to unipartite projections.

## 1. Environment and imports

In [ ]:
# Uncomment to install required packages
# %pip install networkx numpy pandas matplotlib scipy tqdm requests beautifulsoup4

In [ ]:
from __future__ import annotations

import gzip
import json
import re
from collections import deque
from pathlib import Path
from typing import Dict, Iterable, List
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
# Select dataset to process
DATASET_NAME = "DG-AssocMiner"  # Change this to process different BioSNAP datasets

# BioSNAP dataset names
BIOSNAP_NAMES = [
    "CC-Neuron",
    "DD-Miner",
    "ChG-Miner",
    "ChG-InterDecagon",
    "DCh-Miner",
    "ChSe-Decagon",
    "DG-AssocMiner",
    "FF-Miner",
    "DF-Miner",
]

# Output location
OUTPUT_DIR = Path(f"/dccstor/cgq4hls/Q/biosnap/{DATASET_NAME}/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw data location
RAW_DIR = Path(f"/dccstor/cgq4hls/Q/biosnap/{DATASET_NAME}/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Sampling settings
TARGET_SIZES = [2000, 5000, 10000]
REPEATS = 30
BASE_SEED = 42

# Degree-aware sampling heuristic settings
RESTART_PROB = 0.15
FRONTIER_WIDTH = 256
DEGREE_WEIGHT_POWER = 0.5
LOCAL_BFS_EXPANSION = 8

print("Dataset:", DATASET_NAME)
print("Output dir:", OUTPUT_DIR.resolve())
print("Raw dir:", RAW_DIR.resolve())

## 3. Download BioSNAP Dataset

In [ ]:
def find_biosnap_dataset_pages(index_url: str = "https://snap.stanford.edu/biodata/index.html") -> dict[str, str]:
    """Parse BioSNAP index and return mapping: dataset name -> dataset page URL"""
    print("[BioSNAP] Fetching index...")
    r = requests.get(index_url, timeout=60)
    r.raise_for_status()
    
    soup = BeautifulSoup(r.text, "html.parser")
    
    pages: dict[str, str] = {}
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True)
        href = a["href"]
        
        for name in BIOSNAP_NAMES:
            if name.lower() in text.lower() or name.lower() in href.lower():
                pages[name] = urljoin(index_url, href)
    
    return pages


def find_download_links_on_page(page_url: str) -> list[str]:
    """Get likely downloadable files from a BioSNAP dataset page."""
    r = requests.get(page_url, timeout=60)
    r.raise_for_status()
    
    soup = BeautifulSoup(r.text, "html.parser")
    
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(page_url, href)
        
        # BioSNAP pages typically expose downloadable CSV/TSV/TXT archives.
        if re.search(r"\.(csv|tsv|txt|gz|zip|tar\.gz)$", full, flags=re.IGNORECASE):
            links.append(full)
    
    # Deduplicate while preserving order.
    seen = set()
    deduped = []
    for x in links:
        if x not in seen:
            seen.add(x)
            deduped.append(x)
    
    return deduped


def download_file(url: str, dest: Path, timeout: int = 60) -> bool:
    """Download a URL to dest. Returns True on success."""
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[EXISTS] {dest.name}")
        return True
    
    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            if r.status_code != 200:
                print(f"[MISS] {url} -> HTTP {r.status_code}")
                return False
            
            total = int(r.headers.get("content-length", 0))
            with open(dest, "wb") as f, tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc=f"{dest.name}",
            ) as pbar:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        print(f"[OK] {dest.name}")
        return True
    
    except Exception as e:
        print(f"[FAILED] {url}: {e}")
        return False


# Download dataset
print(f"Downloading {DATASET_NAME}...")
try:
    pages = find_biosnap_dataset_pages()
    page = pages.get(DATASET_NAME)
    
    if page is None:
        print(f"[WARNING] Could not find BioSNAP page for {DATASET_NAME}")
        print("Available datasets:", list(pages.keys()))
    else:
        print(f"Page: {page}")
        links = find_download_links_on_page(page)
        
        if not links:
            print("[WARNING] No downloadable file links found.")
        else:
            print(f"Found {len(links)} downloadable files")
            for url in links:
                filename = url.split("/")[-1]
                dest = RAW_DIR / filename
                download_file(url, dest)
                
except Exception as e:
    print(f"[ERROR] Could not download BioSNAP dataset: {e}")

## 4. Load and Parse BioSNAP Dataset

In [ ]:
def load_biosnap_edgelist(filepath: Path) -> nx.Graph:
    """Load BioSNAP edge list file (handles .gz compression and various formats)."""
    edges = []
    
    open_func = gzip.open if filepath.suffix == ".gz" else open
    
    with open_func(filepath, "rt") as f:
        for line in f:
            line = line.strip()
            # Skip comments and empty lines
            if not line or line.startswith("#"):
                continue
            
            # Try different delimiters
            parts = None
            for delimiter in ["\t", ",", " "]:
                parts = line.split(delimiter)
                if len(parts) >= 2:
                    break
            
            if parts and len(parts) >= 2:
                try:
                    # Try to parse as integers first
                    u, v = int(parts[0]), int(parts[1])
                    if u != v:  # Skip self-loops
                        edges.append((u, v))
                except ValueError:
                    # If not integers, use string IDs
                    u, v = parts[0].strip(), parts[1].strip()
                    if u != v:
                        edges.append((u, v))
    
    G = nx.Graph()
    G.add_edges_from(edges)
    return G


# Find the main edge list file
raw_files = list(RAW_DIR.glob("*.txt*")) + list(RAW_DIR.glob("*.csv*")) + list(RAW_DIR.glob("*.tsv*"))
if not raw_files:
    print("[ERROR] No data files found in RAW_DIR")
    print("Please manually download the dataset or check the download step.")
else:
    # Use the first file found (or you can select manually)
    raw_file = raw_files[0]
    print(f"Loading from: {raw_file.name}")
    
    print(f"Loading {DATASET_NAME}...")
    G_full = load_biosnap_edgelist(raw_file)
    
    # Keep largest connected component
    if not nx.is_connected(G_full):
        print("Graph is not connected. Extracting largest connected component...")
        lcc_nodes = max(nx.connected_components(G_full), key=len)
        G_full = G_full.subgraph(lcc_nodes).copy()
    
    # Relabel nodes to 0-indexed integers
    G_full = nx.convert_node_labels_to_integers(G_full, first_label=0)
    
    n_nodes = G_full.number_of_nodes()
    n_edges = G_full.number_of_edges()
    
    print(f"Nodes: {n_nodes:,}")
    print(f"Edges: {n_edges:,}")
    print(f"Connected: {nx.is_connected(G_full)}")

## 5. Graph conversion and sampling helpers

In [ ]:
def graph_degree_array(G: nx.Graph) -> np.ndarray:
    if G.number_of_nodes() == 0:
        return np.array([], dtype=float)
    return np.array([d for _, d in G.degree()], dtype=float)


def materialize_undirected_simple_graph(G: nx.Graph) -> nx.Graph:
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from(G.edges(data=True))
    H.remove_edges_from(nx.selfloop_edges(H))
    return H


def induce_subgraph_by_nodes(G: nx.Graph, nodes: Iterable) -> nx.Graph:
    node_set = set(nodes)
    H = nx.Graph()
    H.add_nodes_from((n, G.nodes[n]) for n in node_set)
    H.add_edges_from((u, v, d) for u, v, d in G.edges(data=True) if u in node_set and v in node_set)
    return H


def keep_largest_connected_component(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    lcc_nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(lcc_nodes).copy()


def weighted_choice_without_replacement(items, weights, k, rng):
    items = list(items)
    weights = np.asarray(weights, dtype=float)
    if len(items) == 0:
        return []
    if np.all(weights <= 0):
        weights = np.ones(len(items), dtype=float)
    weights = np.maximum(weights, 1e-12)
    probs = weights / weights.sum()
    k_eff = min(k, len(items))
    idx = rng.choice(len(items), size=k_eff, replace=False, p=probs)
    return [items[i] for i in idx]


def sample_anchor_node(G: nx.Graph, rng: np.random.Generator, power: float = 0.5):
    nodes = list(G.nodes())
    deg = np.array([G.degree(n) for n in nodes], dtype=float)
    weights = np.power(np.maximum(deg, 1.0), power)
    weights = weights / weights.sum()
    return nodes[int(rng.choice(len(nodes), p=weights))]


def sample_degree_targets(G: nx.Graph, sample_size: int, rng: np.random.Generator) -> np.ndarray:
    deg = graph_degree_array(G)
    if len(deg) == 0:
        return np.zeros(sample_size)
    return rng.choice(deg, size=sample_size, replace=True)


def summarize_graph(G: nx.Graph) -> Dict[str, float]:
    deg = graph_degree_array(G)
    return {
        "num_nodes": int(G.number_of_nodes()),
        "num_edges": int(G.number_of_edges()),
        "avg_degree": float(deg.mean()) if len(deg) else 0.0,
        "median_degree": float(np.median(deg)) if len(deg) else 0.0,
        "max_degree": float(deg.max()) if len(deg) else 0.0,
        "num_connected_components": int(nx.number_connected_components(G)),
    }


def degree_histogram_distance(G_ref: nx.Graph, G_sub: nx.Graph, bins: int = 30) -> float:
    deg_ref = graph_degree_array(G_ref)
    deg_sub = graph_degree_array(G_sub)
    if len(deg_ref) == 0 or len(deg_sub) == 0:
        return float("inf")
    upper = max(float(deg_ref.max()), float(deg_sub.max()), 1.0)
    bin_edges = np.linspace(0.0, upper, bins + 1)
    h_ref, _ = np.histogram(deg_ref, bins=bin_edges, density=True)
    h_sub, _ = np.histogram(deg_sub, bins=bin_edges, density=True)
    return float(np.abs(h_ref - h_sub).sum())


full_summary = summarize_graph(G_full)
print("Full graph summary:")
print(full_summary)

## 6. Connected induced subgraph sampling

### Heuristic

The sampler below uses a degree-aware frontier expansion strategy:

1. Pick an anchor node with probability weighted by degree
2. Grow a connected set using a frontier
3. Score candidate frontier nodes using:
   - internal connectivity to the current sample
   - closeness of node degree to a degree target drawn from the full graph
   - a small exploration / restart mechanism
4. Materialize the **induced** subgraph on the selected nodes
5. Keep the largest connected component if needed

This is not exact degree-sequence matching, but it works well as a practical preprocessing heuristic.

In [ ]:
def connected_degree_aware_subgraph(
    G: nx.Graph,
    target_size: int,
    rng: np.random.Generator,
    restart_prob: float = 0.15,
    degree_weight_power: float = 0.5,
    frontier_width: int = 256,
    local_bfs_expansion: int = 8,
) -> nx.Graph:
    if target_size >= G.number_of_nodes():
        return materialize_undirected_simple_graph(G)

    anchor = sample_anchor_node(G, rng, power=degree_weight_power)
    selected = [anchor]
    selected_set = {anchor}
    frontier = set(G.adj[anchor].keys())
    degree_targets = sample_degree_targets(G, target_size, rng)
    target_ptr = 0

    while len(selected) < target_size:
        if not frontier or rng.random() < restart_prob:
            seed_from_selected = selected[int(rng.integers(0, len(selected)))]
            local_frontier = deque([seed_from_selected])
            steps = 0
            while local_frontier and steps < local_bfs_expansion:
                u = local_frontier.popleft()
                nbrs = list(G.adj[u].keys())
                rng.shuffle(nbrs)
                for v in nbrs:
                    if v not in selected_set:
                        frontier.add(v)
                        local_frontier.append(v)
                steps += 1

        if not frontier:
            remaining = list(set(G.nodes()) - selected_set)
            if not remaining:
                break
            candidate = remaining[int(rng.integers(0, len(remaining)))]
            frontier.add(candidate)

        frontier_list = list(frontier)
        if len(frontier_list) > frontier_width:
            frontier_list = weighted_choice_without_replacement(
                frontier_list,
                [max(G.degree(n), 1) for n in frontier_list],
                frontier_width,
                rng,
            )

        target_degree = degree_targets[min(target_ptr, len(degree_targets) - 1)]
        scores = []
        for node in frontier_list:
            deg = G.degree(node)
            internal_links = sum((nbr in selected_set) for nbr in G.adj[node].keys())
            degree_match = 1.0 / (1.0 + abs(deg - target_degree))
            score = 2.0 * internal_links + degree_match + 0.25 * np.log1p(deg)
            scores.append(score)

        scores = np.asarray(scores, dtype=float)
        if np.all(scores <= 0):
            scores = np.ones_like(scores)
        probs = scores / scores.sum()
        chosen = frontier_list[int(rng.choice(len(frontier_list), p=probs))]

        selected.append(chosen)
        selected_set.add(chosen)
        frontier.discard(chosen)
        frontier.update(v for v in G.adj[chosen].keys() if v not in selected_set)
        target_ptr += 1

    H = induce_subgraph_by_nodes(G, selected_set)
    H = materialize_undirected_simple_graph(H)
    H = keep_largest_connected_component(H)

    while H.number_of_nodes() < target_size and H.number_of_nodes() < G.number_of_nodes():
        current_nodes = set(H.nodes())
        boundary = set()
        for u in current_nodes:
            boundary.update(v for v in G.adj[u].keys() if v not in current_nodes)
        if not boundary:
            break
        boundary = list(boundary)
        rng.shuffle(boundary)
        needed = min(target_size - H.number_of_nodes(), len(boundary))
        current_nodes.update(boundary[:needed])
        H = keep_largest_connected_component(induce_subgraph_by_nodes(G, current_nodes))

    return H

## 7. Export helpers

In [ ]:
def save_graph_edgelist_csv(G: nx.Graph, path: Path):
    edges = np.array(list(G.edges()), dtype=int)
    df = pd.DataFrame(edges, columns=["node1", "node2"])
    df.to_csv(path, index=False)


def write_bundle(
    bundle_dir: Path,
    stem: str,
    G: nx.Graph,
    metadata: Dict,
):
    bundle_dir.mkdir(parents=True, exist_ok=True)
    csv_path = bundle_dir / f"{stem}.csv"
    json_path = bundle_dir / f"{stem}.json"

    save_graph_edgelist_csv(G, csv_path)

    metadata = dict(metadata)
    metadata.update({
        "csv_path": str(csv_path),
    })

    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)

    return {
        "csv": csv_path,
        "json": json_path,
    }

## 8. Sample and export subgraphs

In [ ]:
all_samples = []

for target_size in TARGET_SIZES:
    print(f"\n{'='*60}")
    print(f"Sampling {REPEATS} subgraphs of target size {target_size}")
    print(f"{'='*60}")
    
    for repeat_idx in tqdm(range(REPEATS), desc=f"Size {target_size}"):
        seed = BASE_SEED + 1000 * repeat_idx + target_size
        rng = np.random.default_rng(seed)
        
        # Sample subgraph
        H = connected_degree_aware_subgraph(
            G_full,
            target_size=target_size,
            rng=rng,
            restart_prob=RESTART_PROB,
            degree_weight_power=DEGREE_WEIGHT_POWER,
            frontier_width=FRONTIER_WIDTH,
            local_bfs_expansion=LOCAL_BFS_EXPANSION,
        )

        # Compute quality metrics
        summary = summarize_graph(H)
        degree_hist_l1 = degree_histogram_distance(G_full, H)
        
        # Create metadata
        stem = f"{DATASET_NAME}_n{target_size}_rep{repeat_idx:02d}"
        metadata = {
            "dataset_family": "biosnap",
            "dataset_name": DATASET_NAME,
            "network_id": stem,
            "type": "biosnap_subsample",
            "graph_name": stem,
            "n_nodes": summary["num_nodes"],
            "n_edges": summary["num_edges"],
            "original_num_nodes": n_nodes,
            "sampling": {
                "target_size": int(target_size),
                "effective_target_size": summary["num_nodes"],
                "repeat_idx": int(repeat_idx),
                "seed": int(seed),
                "degree_hist_l1": float(degree_hist_l1),
                "restart_prob": RESTART_PROB,
                "degree_weight_power": DEGREE_WEIGHT_POWER,
                "frontier_width": FRONTIER_WIDTH,
                "local_bfs_expansion": LOCAL_BFS_EXPANSION,
            },
        }
        metadata.update(summary)

        # Write bundle
        subdir = OUTPUT_DIR / f"n{target_size}"
        paths = write_bundle(subdir, stem, H, metadata)
        
        # Record sample info
        all_samples.append({
            "target_size": int(target_size),
            "effective_target_size": summary["num_nodes"],
            "repeat_idx": int(repeat_idx),
            "seed": int(seed),
            "degree_hist_l1": float(degree_hist_l1),
            "num_nodes": summary["num_nodes"],
            "num_edges": summary["num_edges"],
            "avg_degree": summary["avg_degree"],
            "median_degree": summary["median_degree"],
            "max_degree": summary["max_degree"],
            "csv_path": str(paths["csv"]),
            "json_path": str(paths["json"]),
        })

print(f"\n{'='*60}")
print("Sampling complete!")
print(f"{'='*60}")

## 9. Results summary

In [ ]:
results_df = pd.DataFrame(all_samples)
print("\nAll samples:")
print(results_df.head(10))

# Save manifest
manifest_path = OUTPUT_DIR / "manifest.csv"
results_df.to_csv(manifest_path, index=False)
print(f"\nSaved manifest: {manifest_path}")

## 10. Quality diagnostics

In [ ]:
# Degree histogram distance statistics
quality_summary = results_df.groupby("target_size")["degree_hist_l1"].agg(["mean", "std", "min", "max"]).reset_index()
print("\nDegree histogram distance by target size:")
print(quality_summary)

# Size statistics
size_summary = results_df.groupby("target_size").agg({
    "effective_target_size": ["mean", "std"],
    "num_edges": ["mean", "std"],
    "avg_degree": ["mean", "std"],
}).reset_index()
print("\nSize statistics by target:")
print(size_summary)

## 11. Visualize degree distribution comparison

In [ ]:
# Pick best sample from each target size (lowest degree_hist_l1)
best_samples = results_df.sort_values("degree_hist_l1").groupby("target_size").first().reset_index()

# Load and plot degree distributions
plt.figure(figsize=(10, 6))

# Full graph
deg_full = graph_degree_array(G_full)
deg_full = deg_full[deg_full > 0]
xs_full = np.sort(np.unique(deg_full))
ccdf_full = np.array([(deg_full >= x).mean() for x in xs_full])
plt.step(xs_full, ccdf_full, where="post", label="Full graph", linewidth=2)

# Best samples
for _, row in best_samples.iterrows():
    csv_path = Path(row["csv_path"])
    
    # Load sample graph
    df = pd.read_csv(csv_path)
    G_sample = nx.Graph()
    G_sample.add_edges_from(zip(df["node1"], df["node2"]))
    
    deg_sample = graph_degree_array(G_sample)
    deg_sample = deg_sample[deg_sample > 0]
    xs_sample = np.sort(np.unique(deg_sample))
    ccdf_sample = np.array([(deg_sample >= x).mean() for x in xs_sample])
    plt.step(xs_sample, ccdf_sample, where="post", 
             label=f"Sample n={row['target_size']} (L1={row['degree_hist_l1']:.3f})", alpha=0.7)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Degree")
plt.ylabel("CCDF")
plt.title(f"Degree Distribution: {DATASET_NAME} Full Graph vs Samples")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Complete!

The BioSNAP dataset has been preprocessed and exported in QuVINE-compatible format.

### Output Structure:

```
processed/
├── n2000/
│   ├── {dataset}_n2000_rep00.csv
│   ├── {dataset}_n2000_rep00.json
│   └── ... (30 repeats)
├── n5000/
├── n10000/
└── manifest.csv
```